# 06: Final report

Joins the per-record metric CSVs produced by `02`-`05`, aggregates them, and writes a thesis-ready markdown summary to `evaluation/reports/final.md`. Plots referenced from the report live in `evaluation/reports/figures/`.

Sections produced:

1. **Headline**: three numbers per dialect: Execution Accuracy (LDBC only), Result-set F1 (LDBC only), Component F1 (all queries).
2. **Behavioural**: Pass@1, Pass@k, mean iterations, mean duration, total tokens, cost per success.
3. **Structural**: Exact Match + per-component F1.
4. **Distance**: token Levenshtein, token Jaccard, normalised TED.
5. **Stratified**: every primary metric sliced by `difficulty` (easy / medium / hard).
6. **Error taxonomy template**: every failing query listed with empty fields for manual categorisation (schema error / hallucination / direction error / predicate error / projection error / aggregation error / JOIN→path error).

Pass@k bar charts, distance-distribution boxplots, and per-component F1 heatmaps land in `figures/`.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

OUTPUTS_DIR = REPO_ROOT / "evaluation" / "outputs"
REPORTS_DIR = REPO_ROOT / "evaluation" / "reports"
FIG_DIR = REPORTS_DIR / "figures"
REPORTS_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
FINAL_MD = REPORTS_DIR / "final.md"

## Load metric CSVs and join

Each row corresponds to one `(query_id, dialect)` translation. The execution metrics file only has LDBC rows; everything else has both schemas. Left-join on the structural file (which always has every record) and let execution columns be `NaN` for TPC-H.

In [ ]:
beh = pd.read_csv(OUTPUTS_DIR / "metrics_behavioural.csv")
stru = pd.read_csv(OUTPUTS_DIR / "metrics_structural.csv")
dist = pd.read_csv(OUTPUTS_DIR / "metrics_distance.csv")
execm = pd.read_csv(OUTPUTS_DIR / "metrics_execution.csv") if (OUTPUTS_DIR / "metrics_execution.csv").exists() else None

key = ["query_id", "schema_name", "dialect", "difficulty"]
df = beh.merge(stru, on=key, how="outer").merge(dist, on=key, how="outer")
if execm is not None:
    df = df.merge(execm, on=key, how="left")
print(f"Joined rows: {len(df)}  (records seen across notebooks)")
df.head(5)

## Headline numbers per dialect

The 'one-line answer' for the thesis abstract:

In [ ]:
headline = pd.DataFrame()
headline["validation_pass_rate"] = df.groupby("dialect")["validation_passed"].mean()
headline["pass@1"] = df.groupby("dialect")["pass_at_1"].mean()
headline["component_f1"] = df.groupby("dialect")["component_f1_overall"].mean()
headline["normalized_ted"] = df.groupby("dialect")["normalized_ted"].mean()
if execm is not None:
    ldbc = df[df["schema_name"] == "ldbc"]
    headline["execution_accuracy (LDBC)"] = ldbc.groupby("dialect")["execution_accuracy"].mean()
    headline["result_f1 (LDBC)"] = ldbc.groupby("dialect")["result_f1"].mean()
headline

## Stratified tables

In [ ]:
primary_cols = ["validation_passed", "pass_at_1", "component_f1_overall", "normalized_ted"]
if execm is not None:
    primary_cols += ["execution_accuracy", "result_f1"]

by_schema_dialect = df.groupby(["schema_name", "dialect"])[primary_cols].mean()
by_difficulty = df.groupby("difficulty")[primary_cols].mean().reindex(["easy", "medium", "hard"])
print("By schema × dialect:")
display(by_schema_dialect)
print("\nBy difficulty:")
display(by_difficulty)

## Plots

Three figures saved to `evaluation/reports/figures/` and referenced from the markdown report.

In [ ]:
# --- Pass@k bar chart per dialect × difficulty ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, d in zip(axes, ["cypher", "aql"], strict=True):
    sub = df[df["dialect"] == d].groupby("difficulty").agg(
        pass_at_1=("pass_at_1", "mean"),
        pass_at_k=("validation_passed", "mean"),
    ).reindex(["easy", "medium", "hard"])
    sub.plot(kind="bar", ax=ax, ylim=(0, 1.05))
    ax.set_title(f"{d.upper()}")
    ax.set_xlabel("difficulty")
    ax.set_ylabel("fraction")
    ax.tick_params(axis="x", rotation=0)
fig.suptitle("Pass@1 vs Pass@k by difficulty")
fig.tight_layout()
fig.savefig(FIG_DIR / "pass_at_k.png", dpi=120)
plt.show()

In [ ]:
# --- Distance distribution boxplot ----------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
subset = df[df["validation_passed"] == True].copy()  # noqa: E712
subset.boxplot(
    column=["levenshtein", "jaccard", "normalized_ted"], by="dialect", ax=ax,
)
ax.set_ylabel("distance (lower = better)")
fig.suptitle("Distance metrics on validated translations")
fig.tight_layout()
fig.savefig(FIG_DIR / "distance_distribution.png", dpi=120)
plt.show()

In [ ]:
# --- Component F1 heatmap (dialect rows × component cols) -----------------
comp_cols = ["f1_node_labels", "f1_edge_types", "f1_directions", "f1_where",
             "f1_return", "f1_order", "f1_limit", "f1_aggregations"]
heat = df.groupby("dialect")[comp_cols].mean()
fig, ax = plt.subplots(figsize=(10, 2.5))
im = ax.imshow(heat.values, vmin=0, vmax=1, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels([c.replace("f1_", "") for c in heat.columns], rotation=30, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.values[i, j]:.2f}", ha="center", va="center", color="white", fontsize=9)
fig.colorbar(im, ax=ax, label="F1")
ax.set_title("Component F1 per dialect")
fig.tight_layout()
fig.savefig(FIG_DIR / "component_f1.png", dpi=120)
plt.show()

## Failure list for manual error-taxonomy annotation

Every record that either failed validation OR (where execution metrics exist) got Execution Accuracy < 1 is listed below with empty taxonomy fields. Fill these in by hand for the thesis discussion chapter.

In [ ]:
failure_mask = ~df["validation_passed"].astype(bool)
if execm is not None:
    failure_mask = failure_mask | (df["execution_accuracy"].fillna(1.0) < 1.0)
failures = df[failure_mask][["query_id", "schema_name", "dialect", "difficulty",
                              "validation_passed", "component_f1_overall", "normalized_ted"]].copy()
if execm is not None and "execution_accuracy" in df.columns:
    failures["execution_accuracy"] = df.loc[failure_mask, "execution_accuracy"]
failures["category"] = ""  # to be filled in manually
failures["notes"] = ""
failures = failures.sort_values(["schema_name", "dialect", "query_id"]).reset_index(drop=True)
print(f"{len(failures)} failure cases to classify.")
failures.head(20)

## Write the final markdown report

In [ ]:
from tabulate import tabulate


def md_table(df_: pd.DataFrame, floatfmt: str = ".3f") -> str:
    return tabulate(df_.reset_index(), headers="keys", tablefmt="github", floatfmt=floatfmt, showindex=False)


model_name = df["model_name"].iloc[0] if "model_name" in df.columns else "unknown"
total_records = len(df)
validated = int(df["validation_passed"].sum())
total_in = int(df["input_tokens"].sum()) if "input_tokens" in df.columns else 0
total_out = int(df["output_tokens"].sum()) if "output_tokens" in df.columns else 0
total_cost = float(df["cost_usd"].sum()) if "cost_usd" in df.columns else 0.0

parts: list[str] = []
parts.append(f"# rows2graph evaluation report\n")
parts.append(f"Generated: {datetime.now().isoformat(timespec='seconds')}\n")
parts.append(f"Model under evaluation: **{model_name}**\n")
parts.append(f"Total translations: **{total_records}** ({validated} validated)\n")
parts.append(f"Total tokens: **{total_in:,}** input / **{total_out:,}** output, approx **${total_cost:,.2f}** USD\n")

parts.append("\n## Headline\n")
parts.append(md_table(headline) + "\n")

parts.append("\n## Stratified by schema × dialect\n")
parts.append(md_table(by_schema_dialect) + "\n")

parts.append("\n## Stratified by difficulty\n")
parts.append(md_table(by_difficulty) + "\n")

parts.append("\n## Behavioural metrics (per dialect)\n")
behavioural_cols = ["pass_at_1", "validation_passed", "iterations_used", "duration_seconds", "input_tokens", "output_tokens", "cost_usd"]
have = [c for c in behavioural_cols if c in df.columns]
parts.append(md_table(df.groupby("dialect")[have].mean()) + "\n")

parts.append("\n## Component F1 breakdown (per dialect)\n")
parts.append(md_table(heat) + "\n")
parts.append("\n![Component F1 heatmap](figures/component_f1.png)\n")
parts.append("![Pass@k bars](figures/pass_at_k.png)\n")
parts.append("![Distance distributions](figures/distance_distribution.png)\n")

parts.append("\n## Error taxonomy (to be filled in manually)\n")
parts.append(
    "Each failure below needs a category in `{schema_error, hallucination, direction_error, "
    "predicate_error, projection_error, aggregation_error, join_to_path_error, other}` and an "
    "optional note. Edit `metrics_execution.csv` / `metrics_structural.csv` to drive the table "
    "if you want it sortable.\n\n"
)
parts.append(md_table(failures, floatfmt=".2f") + "\n")

parts.append("\n## Out of scope\n")
parts.append(
    "- TPC-H execution-based metrics (no Postgres backing in graphonauts2; future work).\n"
    "- Test Suite Accuracy across multiple data variants.\n"
    "- Embedding-cosine distance.\n"
)

FINAL_MD.write_text("\n".join(parts))
print(f"Wrote {FINAL_MD}")

In [ ]:
# Render the first chunk of the report for a sanity-check preview.
preview = FINAL_MD.read_text().split("\n## Stratified by schema", 1)[0]
print(preview)